<a href="https://colab.research.google.com/github/Phavouredphavour/Pizza-Sales-Analysis-/blob/main/curriculum/phase-2b-sql/weeks-01-08-teaching/week-01-select-where-order-by/01-wednesday/exercises/week-01-wed-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — SELECT, WHERE, ORDER BY, LIMIT: Reading Data
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises (Wednesday)**

Today you practiced four building blocks on the `orders` table: `SELECT`, `WHERE` with equality and `!=`, combining conditions with `OR`, and `ORDER BY`/`LIMIT`. These five questions let you apply them on your own against the real Olist order data.

Each question below is **three cells**:

1. A markdown cell with the task and an **Expected** result.
2. A blank `%%sql qN <<` answer cell — replace `-- Your query here` with your query, aliasing the value the check cell needs (usually `AS n`).
3. A locked check cell (plain Python) — run it right after your query. A ✅ print means your query returned the correct value.

Fill in each `%%sql` cell, then run the check cell below it — a ✅ means your query is correct. Do not edit the check cells. Run the setup cell first, then work top to bottom.

### Setup — run this first
Loads the eight Olist tables into a file-based SQLite database at `/content/olist.db` and connects the `%%sql` cell magic. Because `autopandas` is on, every `%%sql` result comes back as a pandas DataFrame, which is exactly what the check cells inspect below.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 482.8/482.8 kB 17.5 MB/s eta 0:00:00


## Question 1 — Count delivered orders

The `orders` table's `order_status` column tells you what happened to each order. Write a query that counts how many orders have `order_status` equal to `delivered`. Alias your count column as `n`.

**Expected:** 96,478 delivered orders

In [2]:
%%sql q1 <<
-- Your query here
SELECT COUNT(*) AS n
FROM orders
WHERE order_status = 'delivered'

In [3]:
# --- CHECK Q1 — do not edit ---
assert int(q1.iloc[0]['n']) == 96478, "Q1: expected 96,478 delivered orders"
print("✅ Q1 correct")
q1  # show the result of your query

✅ Q1 correct


,n
0,96478


## Question 2 — Count canceled orders

Not every order gets delivered. Write a query that counts how many rows in `orders` have `order_status` equal to `canceled`. Alias your count column as `n`.

**Expected:** 625 canceled orders

In [5]:
%%sql q2 <<
-- Your query here
SELECT COUNT(*) AS n
FROM orders
WHERE order_status = 'canceled'

In [6]:
# --- CHECK Q2 — do not edit ---
assert int(q2.iloc[0]['n']) == 625, "Q2: expected 625 canceled orders"
print("✅ Q2 correct")
q2  # show the result of your query

✅ Q2 correct


,n
0,625


## Question 3 — Orders that are NOT delivered

Instead of listing every non-`delivered` status by name, use the `!=` operator to count every order whose `order_status` is not `delivered` in one shot. Alias your count column as `n`.

**Expected:** 2,963 orders that are not delivered

In [7]:
%%sql q3 <<
-- Your query here
SELECT COUNT(*) AS n
FROM orders
WHERE order_status != 'delivered'

In [8]:
# --- CHECK Q3 — do not edit ---
assert int(q3.iloc[0]['n']) == 2963, "Q3: expected 2,963 orders not delivered"
print("✅ Q3 correct")
q3  # show the result of your query

✅ Q3 correct


,n
0,2963


## Question 4 — Canceled OR unavailable

Combine two conditions on the same column with `OR`: count every order whose `order_status` is `canceled` **or** `unavailable`. Alias your count column as `n`.

**Expected:** 1,234 orders (625 canceled + 609 unavailable)

In [9]:
%%sql q4 <<
-- Your query here
SELECT COUNT(*) AS n
FROM orders
WHERE order_status = 'canceled'
   OR order_status = 'unavailable'

In [10]:
# --- CHECK Q4 — do not edit ---
assert int(q4.iloc[0]['n']) == 1234, "Q4: expected 1,234 canceled-or-unavailable orders"
print("✅ Q4 correct")
q4  # show the result of your query

✅ Q4 correct


,n
0,1234


## Question 5 — Total number of orders

Before filtering anything, it helps to know the size of the table you're working with. Write a query that counts every row in `orders`, with no `WHERE` clause at all. Alias your count column as `n`.

**Expected:** 99,441 total orders

In [11]:
%%sql q5 <<
-- Your query here
SELECT COUNT(*) AS n
FROM orders

In [12]:
# --- CHECK Q5 — do not edit ---
assert int(q5.iloc[0]['n']) == 99441, "Q5: expected 99,441 total orders"
print("✅ Q5 correct")
q5  # show the result of your query

✅ Q5 correct


,n
0,99441


In [ ]:
from google.colab import drive
drive.mount('/content/drive')